# RSI + EMA Momentum Confirmation on SPY
## Strategy Brief
This strategy uses the Relative Strength Index (RSI) and Exponential Moving Average (EMA) to confirm momentum in the SPY ETF. The RSI helps identify overbought or oversold conditions, while the EMA confirms the trend direction. A buy signal is generated when the RSI crosses above a threshold and the price is above the EMA, indicating bullish momentum. Conversely, a sell signal is triggered when the RSI crosses below a threshold and the price is below the EMA, indicating bearish momentum. The strategy aims to capture significant price movements by aligning with the prevailing trend.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our RSI + EMA Momentum Confirmation strategy. These parameters include the lookback periods for the RSI and EMA, as well as the RSI thresholds for generating buy and sell signals.

In [ ]:
RSI_PERIOD = 14
EMA_PERIOD = 50
RSI_OVERBOUGHT = 70
RSI_OVERSOLD = 30
START_DATE = '2010-01-01'
END_DATE = 'today'

## PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance from the start date to today. We will then compute the RSI and EMA indicators and plot them overlaid on the SPY price chart to visualize potential signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute RSI
delta = data['Adj Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))

# Compute EMA
ema = data['Adj Close'].ewm(span=EMA_PERIOD, adjust=False).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Adj Close'], label='SPY Price')
plt.plot(ema, label=f'{EMA_PERIOD}-day EMA', linestyle='--')
plt.title('SPY Price with EMA and RSI')
plt.legend()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(rsi, label='RSI')
plt.axhline(RSI_OVERBOUGHT, color='r', linestyle='--')
plt.axhline(RSI_OVERSOLD, color='g', linestyle='--')
plt.title('RSI')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
In this phase, we define the signal generation logic based on the RSI and EMA indicators. We will create a signal series that indicates long (1) or short (-1) positions based on the conditions defined by the strategy.

In [ ]:
# Generate signals
signals = pd.Series(index=data.index, data=0)

# Buy signal
signals[(rsi < RSI_OVERSOLD) & (data['Adj Close'] > ema)] = 1

# Sell signal
signals[(rsi > RSI_OVERBOUGHT) & (data['Adj Close'] < ema)] = -1

# Positions
positions = signals.shift(1).fillna(0)

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the generated signals and plot the resulting equity curve to visualize the strategy's performance over time.

In [ ]:
# Calculate daily returns
daily_returns = data['Adj Close'].pct_change()

# Strategy returns
strategy_returns = daily_returns * positions

# Equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve of RSI + EMA Momentum Confirmation Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. We will also compare these metrics against a simple buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    # Calculate CAGR
    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1

    # Calculate Sharpe Ratio
    risk_free_rate = 0.01
    excess_returns = strategy_returns - risk_free_rate / 252
    sharpe_ratio = np.sqrt(252) * excess_returns.mean() / excess_returns.std()

    # Calculate Sortino Ratio
    downside_returns = excess_returns[excess_returns < 0]
    sortino_ratio = np.sqrt(252) * excess_returns.mean() / downside_returns.std()

    # Calculate Calmar Ratio
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)

    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Calculate strategy performance
strategy_metrics = calculate_performance_metrics(equity_curve)

# Buy-and-hold performance
buy_and_hold_returns = daily_returns.cumsum()
buy_and_hold_equity_curve = (1 + buy_and_hold_returns).cumprod()
buy_and_hold_metrics = calculate_performance_metrics(buy_and_hold_equity_curve)

# Comparison table
performance_comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})
performance_comparison

## PHASE 6 - Deploy & Monitor
To deploy and monitor the strategy, we create a function that downloads the last 60 days of SPY data, computes the current signal, and prints the suggested position for today.

In [ ]:
def get_current_signal():
    # Download last 60 days of SPY data
    recent_data = yf.download('SPY', period='60d')

    # Compute RSI
    delta = recent_data['Adj Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    # Compute EMA
    ema = recent_data['Adj Close'].ewm(span=EMA_PERIOD, adjust=False).mean()

    # Determine current signal
    if rsi.iloc[-1] < RSI_OVERSOLD and recent_data['Adj Close'].iloc[-1] > ema.iloc[-1]:
        position = 'Long'
    elif rsi.iloc[-1] > RSI_OVERBOUGHT and recent_data['Adj Close'].iloc[-1] < ema.iloc[-1]:
        position = 'Short'
    else:
        position = 'Neutral'

    print(f"Today's position: {position}")

get_current_signal()